### Libraries

In [1]:
# Models deep
import torch
import torch.nn as nn
# torch: optimizadores
import torch.optim as optim

# Data
from torch.utils.data import DataLoader
import pandas as pd

# Numerical
import numpy as np

# Import
import pickle

# sklearn: pipeline
from sklearn.pipeline import Pipeline
# sklearn: accuracy
from sklearn.metrics import accuracy_score



# Local libraries
from utils.torch_lib.TabularTransformer import TabularTransformer, TransactionDataset, TabularTransformerWrapper
from utils.skl_lib.LibPreTabTransformer import MaskedPCA, shift_plus_one


Checking if a faster device is available

In [2]:

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cuda


In [3]:
learning_rate = 1e-3
batch_size = 128
epochs = 10

## Importing data

In [4]:

CSV_TRAIN_PATH = '../../data_ieee/transactions_train.feather'
CSV_TEST_PATH = '../../data_ieee/transactions_test.feather'
PKL_PATH = '../../models/preprocessing_config.pkl'
TARGET_COL = 'isFraud'

train_dataset = TransactionDataset(CSV_TRAIN_PATH, PKL_PATH, target_col=TARGET_COL)
test_dataset = TransactionDataset(CSV_TEST_PATH, PKL_PATH, target_col=TARGET_COL)

loader_train = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
loader_test = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

for x_cat, x_cont, labels in loader_train:
    print(f"Categorical Batch Shape: {x_cat.shape}") 
    print(f"Continuous Batch Shape: {x_cont.shape}")  
    print(f"Labels Shape: {labels.shape}")            
    break

Preprocessing data... this may take a moment.
Preprocessing data... this may take a moment.
Categorical Batch Shape: torch.Size([128, 33])
Continuous Batch Shape: torch.Size([128, 170])
Labels Shape: torch.Size([128])


## Defining testing and training

In [5]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)

    model.train()
    for batch, (X_cat, X_cont, y) in enumerate(dataloader):

        X_cat, X_cont, y = X_cat.to(device), X_cont.to(device), y.to(device)

        pred = model(X_cat, X_cont)
        loss = loss_fn(pred, y)


        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X_cat)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):


    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0


    with torch.no_grad():
        for (X_cat, X_cont, y) in dataloader:

            X_cat, X_cont, y = X_cat.to(device), X_cont.to(device), y.to(device)

            pred = model(X_cat, X_cont)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

## Defining the model

In [6]:
# Geting the dimentions
n_categories = train_dataset.get_n_categories()
n_continuous = train_dataset.num_idx[1]

model = TabularTransformer(n_categories=n_categories, n_continuous = n_continuous, n_classes = 2, embed_dim = 16)
model.to(device)

TabularTransformer(
  (embeddings): ModuleList(
    (0): Embedding(461398, 16)
    (1): Embedding(6, 16)
    (2): Embedding(12822, 16)
    (3-4): 2 x Embedding(6, 16)
    (5): Embedding(61, 16)
    (6): Embedding(62, 16)
    (7-9): 3 x Embedding(4, 16)
    (10): Embedding(5, 16)
    (11-16): 6 x Embedding(4, 16)
    (17): Embedding(5, 16)
    (18): Embedding(4, 16)
    (19): Embedding(5, 16)
    (20-22): 3 x Embedding(4, 16)
    (23): Embedding(77, 16)
    (24): Embedding(126, 16)
    (25): Embedding(238, 16)
    (26): Embedding(6, 16)
    (27-31): 5 x Embedding(4, 16)
    (32): Embedding(1688, 16)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
        )
        (linear1): Linear(in_features=16, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear

## Defining the loss and optimization algorithm

In [7]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce_loss = nn.CrossEntropyLoss(reduction='none')

    def forward(self, inputs, targets):
        ce_loss = self.ce_loss(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        return focal_loss.mean()

In [8]:
loss_fn = FocalLoss(alpha=1, gamma=2).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Training loop

In [9]:

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(loader_train, model, loss_fn, optimizer)
    test_loop(loader_test, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 0.294308  [  128/472432]
loss: 0.008128  [12928/472432]
loss: 0.019097  [25728/472432]
loss: 0.053082  [38528/472432]
loss: 0.037772  [51328/472432]
loss: 0.034224  [64128/472432]
loss: 0.034083  [76928/472432]
loss: 0.024559  [89728/472432]
loss: 0.042504  [102528/472432]
loss: 0.022520  [115328/472432]
loss: 0.031380  [128128/472432]
loss: 0.053482  [140928/472432]
loss: 0.026073  [153728/472432]
loss: 0.041757  [166528/472432]
loss: 0.015206  [179328/472432]
loss: 0.023252  [192128/472432]
loss: 0.046959  [204928/472432]
loss: 0.027793  [217728/472432]
loss: 0.032238  [230528/472432]
loss: 0.044375  [243328/472432]
loss: 0.022850  [256128/472432]
loss: 0.026341  [268928/472432]
loss: 0.043685  [281728/472432]
loss: 0.053317  [294528/472432]
loss: 0.034051  [307328/472432]
loss: 0.014685  [320128/472432]
loss: 0.010734  [332928/472432]
loss: 0.016973  [345728/472432]
loss: 0.017745  [358528/472432]
loss: 0.030727  [371328/472432]
loss: 0.

## Saving and checking 

### Model result (with no importing)

In [10]:

x_cat = test_dataset.Xp_cat[0:1, :].to(device)
x_cont = test_dataset.Xp_cont[0:1, :].to(device)
y = test_dataset.y[0:1].to(device)

model.eval()
with torch.no_grad():
    pred = model(x_cat, x_cont)

print(f"Predicción para el primer elemento: {pred} | Actual : {y}")

Predicción para el primer elemento: tensor([[ 2.0703, -0.9441]], device='cuda:0') | Actual : tensor([0], device='cuda:0')


### Saving the model

In [11]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'config': {
        'n_categories': n_categories,
        'n_continuous': n_continuous,
        'n_classes': 2,
        'embed_dim': 16
    }
}
torch.save(checkpoint, '../../models/TabularTransformer_checkpoint.pth')


### Importing the model

In [12]:
ckpt = torch.load('../../models/TabularTransformer_checkpoint.pth', weights_only=False)


model = TabularTransformer(**ckpt['config']) 
model.load_state_dict(ckpt['model_state_dict'])

<All keys matched successfully>

### Model result (imported)

In [13]:
x_cat = test_dataset.Xp_cat[0:1, :].to(device)
x_cont = test_dataset.Xp_cont[0:1, :].to(device)
y = test_dataset.y[0:1].to(device)
model.to(device)
model.eval()
with torch.no_grad():
    pred = model(x_cat, x_cont)

print(f"Predicción para el primer elemento: {pred} | Actual : {y}")

Predicción para el primer elemento: tensor([[ 2.0703, -0.9441]], device='cuda:0') | Actual : tensor([0], device='cuda:0')


## Wrapping the model in sklearn

In [14]:
wrapped_model = TabularTransformerWrapper(model, batch_size=128)

with open(PKL_PATH, 'rb') as f:
    config = pickle.load(f)

wrapped_model.fitted_ = True
pipeline = Pipeline([
    ('scaler', config['preprocessor']),
    ('classifier', wrapped_model)
])
pipeline.fitted_ = True

In [15]:
df = pd.read_feather(CSV_TEST_PATH)

y = torch.tensor(df[TARGET_COL].values, dtype=torch.long)
X = df.drop(columns=[TARGET_COL])

pipeline.predict(X)

array([0, 0, 0, ..., 0, 0, 0], shape=(118108,))

In [16]:

y_pred = pipeline.predict(X)

acc = accuracy_score(y, y_pred)
print(f"Exactitud del modelo: {acc:.4f}")

Exactitud del modelo: 0.9732
